In [ ]:
pip uninstall numpy tensorflow -y


Found existing installation: numpy 1.24.3
Uninstalling numpy-1.24.3:
  Successfully uninstalled numpy-1.24.3
Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0


In [ ]:
pip install numpy==1.24.3 gensim==4.3.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 1.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pyfume to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.2 MB/s eta 0:00:00
  Created wheel for miniful: filename=miniful-0.0.6-py3-none-any.whl size=3506 sha256=2da4a7f8352ae6f5a6b93c3ff20465dc82caa4b27dbeae5ad40eef2c16533686
  Stored in directory: /root/.cache/pip/wheels/9d/ff/2f/afe4cd56f47de147407705626517d68bea0f3b74eb1fb168e6
Successfully built miniful
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
  Attempting uninstall: gensim
    Found existing installation: 

In [ ]:
pip install --upgrade numpy==1.23.5 tensorflow==2.12.0 gensim


In [ ]:

!pip install -U transformers tokenizers



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.4 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.3
    Uninstalling transformers-4.52.3:
      Successfully uninstalled transformers-4.52.3


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from nltk.stem import WordNetLemmatizer
from nltk.stem.isri import ISRIStemmer
from transformers import AutoTokenizer, AutoModel
import torch
from transformers import BertTokenizer, BertModel
from transformers import AutoTokenizer, AutoModel

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
df = pd.read_csv("sampled_30k.csv")

In [ ]:
df

,specialty_id,name_ar,question_body,cleaned_question,stopword_count,total_words,stopword_ratio
0,91,الطب النفسي,أريد التحدث مع طبيب,اريد التحدث مع طبيب,1,4,0.250000
1,25,تغذية,فيني سمنه,فيني سمنه,0,2,0.000000
2,25,تغذية,اريد نظام رجيم مناسب,اريد نظام رجيم مناسب,0,4,0.000000
3,23,طب عيون,صداع غثيان,صداع غثيان,0,2,0.000000
4,14,جراحة العظام والمفاصل,الم شديد جداً في فالجنب الايسر من الظهر من ٥ ا...,الم شديد جداً في فالجنب الايسر من الظهر من ايا...,5,21,0.238095
...,...,...,...,...,...,...,...
29995,14,جراحة العظام والمفاصل,ركبة نافخه وفيها الم شديد ما الحل,ركبة نافخه وفيها الم شديد ما الحل,1,7,0.142857
29996,25,تغذية,عندي حساسية جتني بعد استخدام نظام عذائي,عندي حساسية جتني بعد استخدام نظام عذايي,1,7,0.142857
29997,14,جراحة العظام والمفاصل,التحدث مع الطبيب,التحدث مع الطبيب,1,3,0.333333
29998,23,طب عيون,Tawuniya حرقة في العينين,حرقة في العينين,1,3,0.333333


In [ ]:
arabic_stopwords = set(stopwords.words('arabic'))
stemmer = ISRIStemmer()

In [ ]:
arabic_stopwords = set(stopwords.words('arabic'))

def clean_text(text):
    if pd.isnull(text):
        return ""

    text = str(text)

    text = re.sub(r'[\u064B-\u065F]', '', text)

    text = re.sub(r"http\S+|www\S+|https\S+", '', text)

    text = re.sub(r'[0-9٠-٩]', '', text)

    text = re.sub(r'[A-Za-z]', '', text)

    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)

    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)

    text = re.sub(r'(.)\1{2,}', r'\1', text)

    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text)


    tokens = [word for word in tokens if word not in arabic_stopwords]

    return ' '.join(tokens)


In [ ]:
df['preprocessed_question_body'] = df['question_body'].apply(clean_text)

In [ ]:
arabic_stopwords = set(stopwords.words('arabic'))

def clean_text(text):
    if pd.isnull(text):
        return ""

    text = str(text)

    text = re.sub(r'[\u064B-\u065F]', '', text)

    text = re.sub(r"http\S+|www\S+|https\S+", '', text)

    text = re.sub(r'[0-9٠-٩]', '', text)

    text = re.sub(r'[A-Za-z]', '', text)

    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)

    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)

    text = re.sub(r'(.)\1{2,}', r'\1', text)

    text = re.sub(r'\s+', ' ', text).strip()

    #
    tokens = word_tokenize(text)



    return ' '.join(tokens)


In [ ]:
df['preprocessed_question_body(with_stop_words)'] = df['question_body'].apply(clean_text)

In [ ]:
df['raw_question_body'] = df['question_body']

# TF_IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_raw = TfidfVectorizer(max_features=5000)
tfidf_with_stopwords = TfidfVectorizer(max_features=5000)
tfidf_no_stopwords = TfidfVectorizer(max_features=5000)

X_raw = tfidf_raw.fit_transform(df['raw_question_body'].fillna(''))
X_with_stopwords = tfidf_with_stopwords.fit_transform(df['preprocessed_question_body(with_stop_words)'].fillna(''))
X_no_stopwords = tfidf_no_stopwords.fit_transform(df['preprocessed_question_body'].fillna(''))

X_raw.shape, X_with_stopwords.shape, X_no_stopwords.shape



((30000, 5000), (30000, 5000), (30000, 5000))

# Word2Vec CBoW

In [ ]:
from gensim.models import Word2Vec

sentences_raw = df['raw_question_body'].fillna('').apply(lambda x: x.split()).tolist()
sentences_with_stopwords = df['preprocessed_question_body(with_stop_words)'].fillna('').apply(lambda x: x.split()).tolist()
sentences_no_stopwords = df['preprocessed_question_body'].fillna('').apply(lambda x: x.split()).tolist()

w2v_model_raw = Word2Vec(sentences=sentences_raw, vector_size=100, window=5, min_count=2, sg=0, workers=4)
w2v_model_with_stopwords = Word2Vec(sentences=sentences_with_stopwords, vector_size=100, window=5, min_count=2, sg=0, workers=4)
w2v_model_no_stopwords = Word2Vec(sentences=sentences_no_stopwords, vector_size=100, window=5, min_count=2, sg=0, workers=4)


In [ ]:
word = 'طبيب'

cbow_embeddings = {
    "Raw Text": w2v_model_raw.wv[word].tolist() if word in w2v_model_raw.wv else "Not found in vocabulary",
    "With Stopwords": w2v_model_with_stopwords.wv[word].tolist() if word in w2v_model_with_stopwords.wv else "Not found in vocabulary",
    "Without Stopwords": w2v_model_no_stopwords.wv[word].tolist() if word in w2v_model_no_stopwords.wv else "Not found in vocabulary"
}

cbow_embeddings


{'Raw Text': [-0.44979214668273926,
  0.7859560251235962,
  0.5863466858863831,
  1.4375934600830078,
  -0.7314749360084534,
  -0.5705626606941223,
  1.0885858535766602,
  1.1384111642837524,
  -0.6426573395729065,
  -0.7987499237060547,
  -0.5061506628990173,
  -2.0340523719787598,
  0.19305120408535004,
  1.434794545173645,
  0.2566198408603668,
  -1.1872881650924683,
  0.6664291620254517,
  -1.31558358669281,
  -0.7355228066444397,
  -2.289834976196289,
  0.13722988963127136,
  1.0618964433670044,
  0.3000309467315674,
  -0.36890432238578796,
  -1.1612809896469116,
  0.20956413447856903,
  -1.0841994285583496,
  -0.9786806702613831,
  -0.7355396747589111,
  0.839512288570404,
  0.978061318397522,
  -0.4172987937927246,
  0.7044470906257629,
  -0.0917203351855278,
  -1.0295007228851318,
  1.9411906003952026,
  -0.1478271484375,
  -1.3675106763839722,
  -1.3311429023742676,
  -1.8468010425567627,
  0.5711919665336609,
  -0.4176351726055145,
  0.17902244627475739,
  -0.1981108039617538

# fastatext

In [ ]:
!pip install scipy==1.10.1 --force-reinstall


  Using cached scipy-1.10.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (58 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 14.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.3
    Uninstalling numpy-1.24.3:
      Successfully uninstalled numpy-1.24.3
  Attempting uninstall: scipy
    Found existing installation: scipy 1.13.1
    Uninstalling scipy-1.13.1:
      Successfully uninstalled scipy-1.13.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.12.0 requires numpy<1.24,>=1.22, but you have numpy 1.26.4 which is incompatible.
flax 0.10.6 requires jax>=0.5.1, but you have jax 0.4.30 which is incompatible.
tf-ker

In [ ]:
!pip uninstall -y scipy gensim


Found existing installation: scipy 1.10.1
Uninstalling scipy-1.10.1:
  Successfully uninstalled scipy-1.10.1
Found existing installation: gensim 4.3.0
Uninstalling gensim-4.3.0:
  Successfully uninstalled gensim-4.3.0


In [ ]:
!pip install numpy==1.24.3 scipy==1.10.1 gensim==4.3.0


  Using cached numpy-1.24.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
  Using cached scipy-1.10.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (58 kB)
  Using cached gensim-4.3.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.4 kB)
Using cached numpy-1.24.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
Using cached scipy-1.10.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (34.1 MB)
Using cached gensim-4.3.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (24.1 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.12.0 requires numpy<1.24,>=1.22, but you have numpy 1.24.3 whi

In [ ]:
from gensim.models import FastText


In [ ]:
sentences_raw = df['raw_question_body'].fillna('').apply(lambda x: x.split()).tolist()
sentences_with_stopwords = df['preprocessed_question_body(with_stop_words)'].fillna('').apply(lambda x: x.split()).tolist()
sentences_no_stopwords = df['preprocessed_question_body'].fillna('').apply(lambda x: x.split()).tolist()


In [ ]:
from gensim.models import FastText

fasttext_raw_full = FastText(sentences=sentences_raw, vector_size=100, window=5, min_count=2, sg=0, bucket=100000, workers=2)
fasttext_with_stopwords_full = FastText(sentences=sentences_with_stopwords, vector_size=100, window=5, min_count=5, sg=0, bucket=100000, workers=2)
fasttext_no_stopwords_full = FastText(sentences=sentences_no_stopwords, vector_size=100, window=5, min_count=5, sg=0, bucket=100000, workers=2)


In [ ]:
avg_vecs_raw_full = []
avg_vecs_with_stopwords_full = []
avg_vecs_no_stopwords_full = []

for tokens in sentences_raw:
    vectors = [fasttext_raw_full.wv[word] for word in tokens if word in fasttext_raw_full.wv]
    avg_vecs_raw_full.append(np.mean(vectors, axis=0) if vectors else np.zeros(100))

for tokens in sentences_with_stopwords:
    vectors = [fasttext_with_stopwords_full.wv[word] for word in tokens if word in fasttext_with_stopwords_full.wv]
    avg_vecs_with_stopwords_full.append(np.mean(vectors, axis=0) if vectors else np.zeros(100))

for tokens in sentences_no_stopwords:
    vectors = [fasttext_no_stopwords_full.wv[word] for word in tokens if word in fasttext_no_stopwords_full.wv]
    avg_vecs_no_stopwords_full.append(np.mean(vectors, axis=0) if vectors else np.zeros(100))


In [ ]:
df_raw_full = pd.DataFrame(avg_vecs_raw_full)
df_with_stopwords_full = pd.DataFrame(avg_vecs_with_stopwords_full)
df_no_stopwords_full = pd.DataFrame(avg_vecs_no_stopwords_full)

df_raw_full.head()
df_with_stopwords_full.head()
df_no_stopwords_full.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.375877,0.379472,0.152219,1.524035,-0.004514,-0.772515,0.391838,-0.458118,0.997978,1.327579,...,0.277045,0.172497,0.428193,0.707150,-0.346885,0.734022,-0.050075,-0.720325,-1.587513,-0.256191
1,-0.022100,0.151697,0.021598,-0.078208,0.094050,-0.127766,0.037211,-0.213814,0.821754,0.526689,...,-0.169562,-0.505190,-0.085227,0.288427,0.029209,0.446960,0.481082,0.025668,-0.252073,0.226145
2,-0.107409,0.621870,-0.245909,0.736014,0.212142,-0.211773,0.369961,-0.242921,0.863651,0.822557,...,0.125457,-0.347317,0.131952,0.548980,-0.210593,0.939390,0.217856,-0.263487,-0.971074,-0.037176
3,-0.209322,-0.799140,1.057032,-0.277011,-0.927988,-0.312793,-0.373939,-0.554863,0.336478,0.440438,...,-0.089868,0.372167,-0.423259,-0.009462,0.377046,-0.619796,0.110358,0.122888,0.203440,0.579000
4,-0.391916,-0.878622,0.976401,-0.257321,-0.642978,0.021635,-0.164205,-0.510677,0.427688,0.633944,...,-0.035440,0.114499,-0.387038,-0.215870,0.263395,-0.660867,-0.159670,0.109939,0.386547,0.608746


# Arabbert

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from tqdm import tqdm

tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02")
model = AutoModel.from_pretrained("aubmindlab/bert-base-arabertv02")
model.eval()

def get_batch_cls_embeddings(texts, batch_size=32):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Processing Batches"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", truncation=True, padding=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        cls_batch = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.extend(cls_batch)
    return all_embeddings


In [ ]:
df['arabert_with_sw_vector'] = get_batch_cls_embeddings(df['cleaned_with_stopwords'].tolist(), batch_size=32)
df['arabert_with_sw_vector'] = df['arabert_with_sw_vector'].apply(lambda x: ','.join(map(str, x)))
df[['arabert_with_sw_vector']].to_csv('arabert_with_stopwords_embeddings.csv', index=False)


Processing Batches: 100%|██████████| 938/938 [1:17:22<00:00,  4.95s/it]


In [ ]:
df['arabert_no_sw_vector'] = get_batch_cls_embeddings(df['cleaned_no_stopwords'].tolist(), batch_size=32)
df['arabert_no_sw_vector'] = df['arabert_no_sw_vector'].apply(lambda x: ','.join(map(str, x)))
df[['arabert_no_sw_vector']].to_csv('arabert_no_stopwords_embeddings.csv', index=False)


Processing Batches: 100%|██████████| 938/938 [1:08:53<00:00,  4.41s/it]


In [ ]:
df['arabert_raw_vector'] = get_batch_cls_embeddings(df['raw_text'].tolist(), batch_size=32)
df['arabert_raw_vector'] = df['arabert_raw_vector'].apply(lambda x: ','.join(map(str, x)))
df[['arabert_raw_vector']].to_csv('arabert_raw_embeddings.csv', index=False)

Processing Batches: 100%|██████████| 938/938 [18:47<00:00,  1.20s/it]


# Logistic refression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report


In [ ]:
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(df['name_ar'])

TF-IDF LR

In [ ]:
tfidf_raw = TfidfVectorizer(max_features=5000)
tfidf_with_sw = TfidfVectorizer(max_features=5000)
tfidf_no_sw = TfidfVectorizer(max_features=5000)

In [ ]:

X_raw = tfidf_raw.fit_transform(df['raw_question_body'].fillna(''))
X_with_sw = tfidf_with_sw.fit_transform(df['preprocessed_question_body(with_stop_words)'].fillna(''))
X_no_sw = tfidf_no_sw.fit_transform(df['preprocessed_question_body'].fillna(''))


In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, encoded_labels, test_size=0.2, random_state=42)
X_train_with_sw, X_test_with_sw, _, _ = train_test_split(X_with_sw, encoded_labels, test_size=0.2, random_state=42)
X_train_no_sw, X_test_no_sw, _, _ = train_test_split(X_no_sw, encoded_labels, test_size=0.2, random_state=42)


In [ ]:
lr_raw = LogisticRegression(max_iter=1000)
lr_raw.fit(X_train_raw, y_train)
y_pred_raw = lr_raw.predict(X_test_raw)
acc_raw = accuracy_score(y_test, y_pred_raw)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision_raw = precision_score(y_test, y_pred_raw, average='weighted')
recall_raw = recall_score(y_test, y_pred_raw, average='weighted')
f1_raw = f1_score(y_test, y_pred_raw, average='weighted')

{
    "Accuracy": acc_raw,
    "Precision": precision_raw,
    "Recall": recall_raw,
    "F1 Score": f1_raw
}

{'Accuracy': 0.8701666666666666,
 'Precision': 0.8790170714197317,
 'Recall': 0.8701666666666666,
 'F1 Score': 0.8720512248871652}

In [ ]:
lr_with_sw = LogisticRegression(max_iter=1000)
lr_with_sw.fit(X_train_with_sw, y_train)
y_pred_with_sw = lr_with_sw.predict(X_test_with_sw)
acc_with_sw = accuracy_score(y_test, y_pred_with_sw)

In [ ]:
acc_with_sw = accuracy_score(y_test, y_pred_with_sw)
precision_with_sw = precision_score(y_test, y_pred_with_sw, average='weighted')
recall_with_sw = recall_score(y_test, y_pred_with_sw, average='weighted')
f1_with_sw = f1_score(y_test, y_pred_with_sw, average='weighted')

{
    "Accuracy": acc_with_sw,
    "Precision": precision_with_sw,
    "Recall": recall_with_sw,
    "F1 Score": f1_with_sw
}

{'Accuracy': 0.8715,
 'Precision': 0.8805953007744293,
 'Recall': 0.8715,
 'F1 Score': 0.8734175193780018}

In [ ]:
lr_no_sw = LogisticRegression(max_iter=1000)
lr_no_sw.fit(X_train_no_sw, y_train)
y_pred_no_sw = lr_no_sw.predict(X_test_no_sw)
acc_no_sw = accuracy_score(y_test, y_pred_no_sw)


In [ ]:
acc_no_sw = accuracy_score(y_test, y_pred_no_sw)
precision_no_sw = precision_score(y_test, y_pred_no_sw, average='weighted')
recall_no_sw = recall_score(y_test, y_pred_no_sw, average='weighted')
f1_no_sw = f1_score(y_test, y_pred_no_sw, average='weighted')

{
    "Accuracy": acc_no_sw,
    "Precision": precision_no_sw,
    "Recall": recall_no_sw,
    "F1 Score": f1_no_sw
}

{'Accuracy': 0.8735,
 'Precision': 0.8821697956187906,
 'Recall': 0.8735,
 'F1 Score': 0.8752868929555769}

CBoW LR

In [ ]:
cbow_raw_embeddings = []
for tokens in sentences_raw:
    vectors = [w2v_model_raw.wv[word] for word in tokens if word in w2v_model_raw.wv]
    if vectors:
        avg_vector = np.mean(vectors, axis=0)
    else:
        avg_vector = np.zeros(w2v_model_raw.vector_size)
    cbow_raw_embeddings.append(avg_vector)

X_cbow_raw = np.array(cbow_raw_embeddings)

le = LabelEncoder()
y = le.fit_transform(df['name_ar'])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_cbow_raw, y, test_size=0.2, random_state=42, stratify=y)

lr_cbow_raw = LogisticRegression(max_iter=1000)
lr_cbow_raw.fit(X_train, y_train)
y_pred = lr_cbow_raw.predict(X_test)

In [ ]:
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

{
    "Accuracy": acc,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1
}

{'Accuracy': 0.7466666666666667,
 'Precision': 0.7598694512520977,
 'Recall': 0.7466666666666667,
 'F1 Score': 0.7483285467034728}

In [ ]:
sentences_with_stopwords = df['preprocessed_question_body(with_stop_words)'].fillna('').apply(lambda x: x.split()).tolist()

cbow_with_sw_embeddings = []
for tokens in sentences_with_stopwords:
    vectors = [w2v_model_with_stopwords.wv[word] for word in tokens if word in w2v_model_with_stopwords.wv]
    if vectors:
        avg_vector = np.mean(vectors, axis=0)
    else:
        avg_vector = np.zeros(w2v_model_with_stopwords.vector_size)
    cbow_with_sw_embeddings.append(avg_vector)

X_cbow_with_sw = np.array(cbow_with_sw_embeddings)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_cbow_with_sw, y, test_size=0.2, random_state=42, stratify=y)

lr_cbow_with_sw = LogisticRegression(max_iter=1000)
lr_cbow_with_sw.fit(X_train, y_train)
y_pred = lr_cbow_with_sw.predict(X_test)

In [ ]:
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

{
    "Accuracy": acc,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1
}

{'Accuracy': 0.751,
 'Precision': 0.7641761676261534,
 'Recall': 0.751,
 'F1 Score': 0.7526164852180004}

In [ ]:
sentences_no_stopwords = df['preprocessed_question_body'].fillna('').apply(lambda x: x.split()).tolist()

cbow_no_sw_embeddings = []
for tokens in sentences_no_stopwords:
    vectors = [w2v_model_no_stopwords.wv[word] for word in tokens if word in w2v_model_no_stopwords.wv]
    if vectors:
        avg_vector = np.mean(vectors, axis=0)
    else:
        avg_vector = np.zeros(w2v_model_no_stopwords.vector_size)
    cbow_no_sw_embeddings.append(avg_vector)

X_cbow_no_sw = np.array(cbow_no_sw_embeddings)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_cbow_no_sw, y, test_size=0.2, random_state=42, stratify=y)

lr_cbow_no_sw = LogisticRegression(max_iter=1000)
lr_cbow_no_sw.fit(X_train, y_train)
y_pred = lr_cbow_no_sw.predict(X_test)


In [ ]:
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

{
    "Accuracy": acc,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1
}

{'Accuracy': 0.7706666666666667,
 'Precision': 0.7838446722417852,
 'Recall': 0.7706666666666667,
 'F1 Score': 0.7727078297462396}

Fasttext LR

In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df['name_ar'])

X_train_with_sw, X_test_with_sw, y_train, y_test = train_test_split(
    df_with_stopwords_full, y_encoded, test_size=0.2, random_state=42
)

lr_fasttext_with_sw = LogisticRegression(max_iter=1000)
lr_fasttext_with_sw.fit(X_train_with_sw, y_train)

y_pred_with_sw = lr_fasttext_with_sw.predict(X_test_with_sw)

In [ ]:
acc_with_sw = accuracy_score(y_test, y_pred_with_sw)
precision_with_sw = precision_score(y_test, y_pred_with_sw, average='weighted')
recall_with_sw = recall_score(y_test, y_pred_with_sw, average='weighted')
f1_with_sw = f1_score(y_test, y_pred_with_sw, average='weighted')

{
    "Accuracy": acc_with_sw,
    "Precision": precision_with_sw,
    "Recall": recall_with_sw,
    "F1 Score": f1_with_sw
}

{'Accuracy': 0.7825,
 'Precision': 0.7899090934200307,
 'Recall': 0.7825,
 'F1 Score': 0.7839559470271368}

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df_raw_full, y_encoded, test_size=0.2, random_state=42
)

lr_fasttext_raw = LogisticRegression(max_iter=1000)
lr_fasttext_raw.fit(X_train_raw, y_train)

y_pred_raw = lr_fasttext_raw.predict(X_test_raw)

In [ ]:
acc_raw = accuracy_score(y_test, y_pred_raw)
precision_raw = precision_score(y_test, y_pred_raw, average='weighted')
recall_raw = recall_score(y_test, y_pred_raw, average='weighted')
f1_raw = f1_score(y_test, y_pred_raw, average='weighted')

{
    "Accuracy": acc_raw,
    "Precision": precision_raw,
    "Recall": recall_raw,
    "F1 Score": f1_raw
}

{'Accuracy': 0.7708333333333334,
 'Precision': 0.7802550525275901,
 'Recall': 0.7708333333333334,
 'F1 Score': 0.7726995334722782}

In [ ]:
X_train_no_sw, X_test_no_sw, y_train, y_test = train_test_split(
    df_no_stopwords_full, y_encoded, test_size=0.2, random_state=42
)

lr_fasttext_no_sw = LogisticRegression(max_iter=1000)
lr_fasttext_no_sw.fit(X_train_no_sw, y_train)

y_pred_no_sw = lr_fasttext_no_sw.predict(X_test_no_sw)

In [ ]:
acc_no_sw = accuracy_score(y_test, y_pred_no_sw)
precision_no_sw = precision_score(y_test, y_pred_no_sw, average='weighted')
recall_no_sw = recall_score(y_test, y_pred_no_sw, average='weighted')
f1_no_sw = f1_score(y_test, y_pred_no_sw, average='weighted')

{
    "Accuracy": acc_no_sw,
    "Precision": precision_no_sw,
    "Recall": recall_no_sw,
    "F1 Score": f1_no_sw
}

{'Accuracy': 0.7991666666666667,
 'Precision': 0.8062532194934684,
 'Recall': 0.7991666666666667,
 'F1 Score': 0.800579534773352}

Arabbert LR

In [ ]:
df_arabert_raw = pd.read_csv('arabert_raw_embeddings.csv')

arabert_raw_vectors = df_arabert_raw['arabert_raw_vector'].apply(
    lambda x: np.fromstring(x, sep=',')
).tolist()

X_arabert_raw = np.vstack(arabert_raw_vectors)

X_train_arabert_raw, X_test_arabert_raw, y_train, y_test = train_test_split(
    X_arabert_raw, y_encoded, test_size=0.2, random_state=42
)

lr_arabert_raw = LogisticRegression(max_iter=1000)
lr_arabert_raw.fit(X_train_arabert_raw, y_train)

y_pred_arabert_raw = lr_arabert_raw.predict(X_test_arabert_raw)

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
acc_arabert_raw = accuracy_score(y_test, y_pred_arabert_raw)
precision_arabert_raw = precision_score(y_test, y_pred_arabert_raw, average='weighted')
recall_arabert_raw = recall_score(y_test, y_pred_arabert_raw, average='weighted')
f1_arabert_raw = f1_score(y_test, y_pred_arabert_raw, average='weighted')

{
    "Accuracy": acc_arabert_raw,
    "Precision": precision_arabert_raw,
    "Recall": recall_arabert_raw,
    "F1 Score": f1_arabert_raw
}

{'Accuracy': 0.8546666666666667,
 'Precision': 0.8563681866767884,
 'Recall': 0.8546666666666667,
 'F1 Score': 0.8552071651949433}

In [ ]:
df_arabert_with_sw = pd.read_csv('arabert_with_stopwords_embeddings.csv')

arabert_with_sw_vectors = df_arabert_with_sw['arabert_with_sw_vector'].apply(
    lambda x: np.fromstring(x, sep=',')
).tolist()

X_arabert_with_sw = np.vstack(arabert_with_sw_vectors)

X_train_arabert_with_sw, X_test_arabert_with_sw, y_train, y_test = train_test_split(
    X_arabert_with_sw, y_encoded, test_size=0.2, random_state=42
)

lr_arabert_with_sw = LogisticRegression(max_iter=1000)
lr_arabert_with_sw.fit(X_train_arabert_with_sw, y_train)

y_pred_arabert_with_sw = lr_arabert_with_sw.predict(X_test_arabert_with_sw)

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
acc_arabert_with_sw = accuracy_score(y_test, y_pred_arabert_with_sw)
precision_arabert_with_sw = precision_score(y_test, y_pred_arabert_with_sw, average='weighted')
recall_arabert_with_sw = recall_score(y_test, y_pred_arabert_with_sw, average='weighted')
f1_arabert_with_sw = f1_score(y_test, y_pred_arabert_with_sw, average='weighted')

{
    "Accuracy": acc_arabert_with_sw,
    "Precision": precision_arabert_with_sw,
    "Recall": recall_arabert_with_sw,
    "F1 Score": f1_arabert_with_sw
}

{'Accuracy': 0.8585,
 'Precision': 0.8603202528613311,
 'Recall': 0.8585,
 'F1 Score': 0.859050560080812}

In [ ]:
df_arabert_no_sw = pd.read_csv('arabert_no_stopwords_embeddings.csv')

arabert_no_sw_vectors = df_arabert_no_sw['arabert_no_sw_vector'].apply(
    lambda x: np.fromstring(x, sep=',')
).tolist()

X_arabert_no_sw = np.vstack(arabert_no_sw_vectors)

X_train_arabert_no_sw, X_test_arabert_no_sw, y_train, y_test = train_test_split(
    X_arabert_no_sw, y_encoded, test_size=0.2, random_state=42
)

lr_arabert_no_sw = LogisticRegression(max_iter=1000)
lr_arabert_no_sw.fit(X_train_arabert_no_sw, y_train)

y_pred_arabert_no_sw = lr_arabert_no_sw.predict(X_test_arabert_no_sw)



In [ ]:

acc_arabert_no_sw = accuracy_score(y_test, y_pred_arabert_no_sw)
precision_arabert_no_sw = precision_score(y_test, y_pred_arabert_no_sw, average='weighted')
recall_arabert_no_sw = recall_score(y_test, y_pred_arabert_no_sw, average='weighted')
f1_arabert_no_sw = f1_score(y_test, y_pred_arabert_no_sw, average='weighted')


{
    "Accuracy": acc_arabert_no_sw,
    "Precision": precision_arabert_no_sw,
    "Recall": recall_arabert_no_sw,
    "F1 Score": f1_arabert_no_sw
}

{'Accuracy': 0.8568333333333333,
 'Precision': 0.858814688730496,
 'Recall': 0.8568333333333333,
 'F1 Score': 0.8574543268008816}

# Naive Bayes

TF-IDF NB

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

In [ ]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(X_raw.toarray(), y_encoded, test_size=0.2, random_state=42)

nb_raw = GaussianNB()
nb_raw.fit(X_train_raw, y_train_raw)

GaussianNB()

In [ ]:
y_pred_raw = nb_raw.predict(X_test_raw)
print("Naive Bayes - TF-IDF Raw Embeddings Evaluation")
print(classification_report(y_test_raw, y_pred_raw))

Naive Bayes - TF-IDF Raw Embeddings Evaluation
              precision    recall  f1-score   support

           0       0.76      0.54      0.63      1212
           1       0.43      0.87      0.57       882
           2       0.88      0.60      0.71      1617
           3       0.85      0.81      0.83      1243
           4       0.76      0.78      0.77      1046

    accuracy                           0.70      6000
   macro avg       0.74      0.72      0.70      6000
weighted avg       0.76      0.70      0.71      6000



In [ ]:
X_train_with_sw, X_test_with_sw, y_train_with_sw, y_test_with_sw = train_test_split(
    X_with_sw.toarray(), y_encoded, test_size=0.2, random_state=42
)

nb_with_sw = GaussianNB()
nb_with_sw.fit(X_train_with_sw, y_train_with_sw)

GaussianNB()

In [ ]:
y_pred_with_sw = nb_with_sw.predict(X_test_with_sw)
print("Naive Bayes - TF-IDF With Stopwords Evaluation")
print(classification_report(y_test_with_sw, y_pred_with_sw))

Naive Bayes - TF-IDF With Stopwords Evaluation
              precision    recall  f1-score   support

           0       0.77      0.54      0.64      1212
           1       0.41      0.86      0.56       882
           2       0.88      0.59      0.70      1617
           3       0.86      0.79      0.82      1243
           4       0.76      0.77      0.77      1046

    accuracy                           0.69      6000
   macro avg       0.74      0.71      0.70      6000
weighted avg       0.76      0.69      0.71      6000



In [ ]:
X_train_no_sw, X_test_no_sw, y_train_no_sw, y_test_no_sw = train_test_split(
    X_no_sw.toarray(), y_encoded, test_size=0.2, random_state=42
)

nb_no_sw = GaussianNB()
nb_no_sw.fit(X_train_no_sw, y_train_no_sw)

GaussianNB()

In [ ]:
y_pred_no_sw = nb_no_sw.predict(X_test_no_sw)
print("Naive Bayes - TF-IDF Without Stopwords Evaluation")
print(classification_report(y_test_no_sw, y_pred_no_sw))

Naive Bayes - TF-IDF Without Stopwords Evaluation
              precision    recall  f1-score   support

           0       0.77      0.54      0.64      1212
           1       0.42      0.87      0.56       882
           2       0.89      0.59      0.71      1617
           3       0.85      0.80      0.82      1243
           4       0.76      0.77      0.77      1046

    accuracy                           0.70      6000
   macro avg       0.74      0.71      0.70      6000
weighted avg       0.76      0.70      0.71      6000



CBoW NB

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

X_train_cbow_raw, X_test_cbow_raw, y_train_cbow_raw, y_test_cbow_raw = train_test_split(
    cbow_raw_embeddings, y_encoded, test_size=0.2, random_state=42
)

nb_cbow_raw = GaussianNB()
nb_cbow_raw.fit(X_train_cbow_raw, y_train_cbow_raw)


GaussianNB()

In [ ]:
y_pred_cbow_raw = nb_cbow_raw.predict(X_test_cbow_raw)
print("Naive Bayes - CBoW Raw Text Evaluation")
print(classification_report(y_test_cbow_raw, y_pred_cbow_raw))

Naive Bayes - CBoW Raw Text Evaluation
              precision    recall  f1-score   support

           0       0.43      0.69      0.53      1212
           1       0.71      0.55      0.62       882
           2       0.65      0.51      0.57      1617
           3       0.77      0.46      0.58      1243
           4       0.46      0.60      0.52      1046

    accuracy                           0.56      6000
   macro avg       0.60      0.56      0.56      6000
weighted avg       0.61      0.56      0.56      6000



In [ ]:
X_train_cbow_with_sw, X_test_cbow_with_sw, y_train_cbow_with_sw, y_test_cbow_with_sw = train_test_split(
    X_cbow_with_sw, y_encoded, test_size=0.2, random_state=42
)

nb_model_cbow_with_sw = GaussianNB()
nb_model_cbow_with_sw.fit(X_train_cbow_with_sw, y_train_cbow_with_sw)


GaussianNB()

In [ ]:

y_pred_cbow_with_sw = nb_model_cbow_with_sw.predict(X_test_cbow_with_sw)

print("Naive Bayes - CBoW with Stopwords Embeddings")
print(classification_report(y_test_cbow_with_sw, y_pred_cbow_with_sw))

Naive Bayes - CBoW with Stopwords Embeddings
              precision    recall  f1-score   support

           0       0.45      0.77      0.57      1212
           1       0.70      0.59      0.64       882
           2       0.66      0.51      0.57      1617
           3       0.79      0.52      0.62      1243
           4       0.50      0.54      0.52      1046

    accuracy                           0.58      6000
   macro avg       0.62      0.58      0.59      6000
weighted avg       0.62      0.58      0.58      6000



In [ ]:
X_train_cbow_no_sw, X_test_cbow_no_sw, y_train_cbow_no_sw, y_test_cbow_no_sw = train_test_split(
    X_cbow_no_sw, y_encoded, test_size=0.2, random_state=42
)

nb_model_cbow_no_sw = GaussianNB()
nb_model_cbow_no_sw.fit(X_train_cbow_no_sw, y_train_cbow_no_sw)

GaussianNB()

In [ ]:
y_pred_cbow_no_sw = nb_model_cbow_no_sw.predict(X_test_cbow_no_sw)

print("Naive Bayes - CBoW Without Stopwords Embeddings")
print(classification_report(y_test_cbow_no_sw, y_pred_cbow_no_sw))

Naive Bayes - CBoW Without Stopwords Embeddings
              precision    recall  f1-score   support

           0       0.41      0.81      0.55      1212
           1       0.80      0.57      0.67       882
           2       0.72      0.48      0.58      1617
           3       0.87      0.51      0.64      1243
           4       0.50      0.56      0.53      1046

    accuracy                           0.58      6000
   macro avg       0.66      0.59      0.59      6000
weighted avg       0.66      0.58      0.59      6000



Fasttext NB

In [ ]:
X_fasttext_raw = df_raw_full.values

X_train_ft_raw, X_test_ft_raw, y_train_ft_raw, y_test_ft_raw = train_test_split(
    X_fasttext_raw, y_encoded, test_size=0.2, random_state=42
)

nb_model_ft_raw = GaussianNB()
nb_model_ft_raw.fit(X_train_ft_raw, y_train_ft_raw)

GaussianNB()

In [ ]:
y_pred_ft_raw = nb_model_ft_raw.predict(X_test_ft_raw)

print("Naive Bayes - FastText Raw Embeddings")
print(classification_report(y_test_ft_raw, y_pred_ft_raw))

Naive Bayes - FastText Raw Embeddings
              precision    recall  f1-score   support

           0       0.43      0.78      0.56      1212
           1       0.69      0.58      0.63       882
           2       0.62      0.48      0.54      1617
           3       0.85      0.40      0.54      1243
           4       0.48      0.57      0.52      1046

    accuracy                           0.55      6000
   macro avg       0.61      0.56      0.56      6000
weighted avg       0.62      0.55      0.55      6000



In [ ]:

X_fasttext_with_sw = df_with_stopwords_full.values


X_train_ft_with_sw, X_test_ft_with_sw, y_train_ft_with_sw, y_test_ft_with_sw = train_test_split(
    X_fasttext_with_sw, y_encoded, test_size=0.2, random_state=42
)

nb_model_ft_with_sw = GaussianNB()
nb_model_ft_with_sw.fit(X_train_ft_with_sw, y_train_ft_with_sw)


GaussianNB()

In [ ]:

y_pred_ft_with_sw = nb_model_ft_with_sw.predict(X_test_ft_with_sw)


print("Naive Bayes - FastText With Stopwords Embeddings")
print(classification_report(y_test_ft_with_sw, y_pred_ft_with_sw))

Naive Bayes - FastText With Stopwords Embeddings
              precision    recall  f1-score   support

           0       0.46      0.78      0.58      1212
           1       0.69      0.59      0.63       882
           2       0.66      0.52      0.58      1617
           3       0.85      0.51      0.64      1243
           4       0.55      0.62      0.58      1046

    accuracy                           0.60      6000
   macro avg       0.64      0.60      0.60      6000
weighted avg       0.64      0.60      0.60      6000



In [ ]:
X_fasttext_no_sw = df_no_stopwords_full.values

X_train_ft_no_sw, X_test_ft_no_sw, y_train_ft_no_sw, y_test_ft_no_sw = train_test_split(
    X_fasttext_no_sw, y_encoded, test_size=0.2, random_state=42
)

nb_model_ft_no_sw = GaussianNB()
nb_model_ft_no_sw.fit(X_train_ft_no_sw, y_train_ft_no_sw)

GaussianNB()

In [ ]:

y_pred_ft_no_sw = nb_model_ft_no_sw.predict(X_test_ft_no_sw)

print("Naive Bayes - FastText Without Stopwords Embeddings")
print(classification_report(y_test_ft_no_sw, y_pred_ft_no_sw))

Naive Bayes - FastText Without Stopwords Embeddings
              precision    recall  f1-score   support

           0       0.47      0.83      0.60      1212
           1       0.78      0.61      0.68       882
           2       0.74      0.52      0.61      1617
           3       0.92      0.55      0.69      1243
           4       0.52      0.65      0.58      1046

    accuracy                           0.62      6000
   macro avg       0.69      0.63      0.63      6000
weighted avg       0.69      0.62      0.63      6000



In [ ]:
arabert_raw_df = pd.read_csv("arabert_raw_embeddings.csv")

arabert_raw_vectors = arabert_raw_df.iloc[:, 0].apply(lambda x: np.fromstring(x, sep=','))

X_arabert_raw = np.stack(arabert_raw_vectors.values)

X_train_arabert_raw, X_test_arabert_raw, y_train_arabert_raw, y_test_arabert_raw = train_test_split(
    X_arabert_raw, y_encoded, test_size=0.2, random_state=42
)

nb_model_arabert_raw = GaussianNB()
nb_model_arabert_raw.fit(X_train_arabert_raw, y_train_arabert_raw)

GaussianNB()

In [ ]:

y_pred_arabert_raw = nb_model_arabert_raw.predict(X_test_arabert_raw)

print("Naive Bayes - AraBERT Raw Embeddings")
print(classification_report(y_test_arabert_raw, y_pred_arabert_raw))

Naive Bayes - AraBERT Raw Embeddings
              precision    recall  f1-score   support

           0       0.65      0.80      0.72      1212
           1       0.83      0.79      0.81       882
           2       0.73      0.70      0.71      1617
           3       0.71      0.66      0.68      1243
           4       0.64      0.60      0.62      1046

    accuracy                           0.71      6000
   macro avg       0.71      0.71      0.71      6000
weighted avg       0.71      0.71      0.70      6000



In [ ]:
arabert_with_sw_df = pd.read_csv("arabert_with_stopwords_embeddings.csv")

arabert_with_sw_vectors = arabert_with_sw_df.iloc[:, 0].apply(lambda x: np.fromstring(x, sep=','))

X_arabert_with_sw = np.stack(arabert_with_sw_vectors.values)

X_train_arabert_with_sw, X_test_arabert_with_sw, y_train_arabert_with_sw, y_test_arabert_with_sw = train_test_split(
    X_arabert_with_sw, y_encoded, test_size=0.2, random_state=42
)

nb_model_arabert_with_sw = GaussianNB()
nb_model_arabert_with_sw.fit(X_train_arabert_with_sw, y_train_arabert_with_sw)


GaussianNB()

In [ ]:
y_pred_arabert_with_sw = nb_model_arabert_with_sw.predict(X_test_arabert_with_sw)


print("Naive Bayes - AraBERT Embeddings (With Stopwords)")
print(classification_report(y_test_arabert_with_sw, y_pred_arabert_with_sw))

Naive Bayes - AraBERT Embeddings (With Stopwords)
              precision    recall  f1-score   support

           0       0.64      0.78      0.70      1212
           1       0.82      0.76      0.79       882
           2       0.71      0.69      0.70      1617
           3       0.70      0.66      0.68      1243
           4       0.63      0.58      0.60      1046

    accuracy                           0.69      6000
   macro avg       0.70      0.69      0.69      6000
weighted avg       0.70      0.69      0.69      6000



In [ ]:
arabert_no_sw_df = pd.read_csv("arabert_no_stopwords_embeddings.csv")

arabert_no_sw_vectors = arabert_no_sw_df.iloc[:, 0].apply(lambda x: np.fromstring(x, sep=','))

X_arabert_no_sw = np.stack(arabert_no_sw_vectors.values)

X_train_arabert_no_sw, X_test_arabert_no_sw, y_train_arabert_no_sw, y_test_arabert_no_sw = train_test_split(
    X_arabert_no_sw, y_encoded, test_size=0.2, random_state=42
)

nb_model_arabert_no_sw = GaussianNB()
nb_model_arabert_no_sw.fit(X_train_arabert_no_sw, y_train_arabert_no_sw)

GaussianNB()

In [ ]:

y_pred_arabert_no_sw = nb_model_arabert_no_sw.predict(X_test_arabert_no_sw)


print("Naive Bayes - AraBERT Embeddings (Without Stopwords)")
print(classification_report(y_test_arabert_no_sw, y_pred_arabert_no_sw))

Naive Bayes - AraBERT Embeddings (Without Stopwords)
              precision    recall  f1-score   support

           0       0.66      0.80      0.72      1212
           1       0.83      0.78      0.80       882
           2       0.73      0.69      0.71      1617
           3       0.72      0.68      0.70      1243
           4       0.64      0.62      0.63      1046

    accuracy                           0.71      6000
   macro avg       0.72      0.71      0.71      6000
weighted avg       0.71      0.71      0.71      6000



16

In [ ]:
!pip install fasttext


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp311-cp311-linux_x86_64.whl size=4313507 sha256=9cb2488ace007155a533dd9f375c1fa815e8e2ab80a613e4852a27b64666d5d3
  Stored in directory: /root/.cache/pip/wheels/65/4f/35/5057db0249224e9ab55a513fa6b79451473ceb7713017823c3
Successfully built fasttext


In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout, Input
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import fasttext

13

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['preprocessed_question_body'].fillna(''))


sequences = tokenizer.texts_to_sequences(df['preprocessed_question_body'].fillna(''))


0

In [ ]:

max_len = max(len(seq) for seq in sequences)


X_padded = pad_sequences(sequences, maxlen=max_len, padding='post')


vocab_size = len(tokenizer.word_index) + 1


0

In [ ]:
!pip install fasttext


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 5.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp311-cp311-linux_x86_64.whl size=4313499 sha256=a0b7e25bc8697422165a6766685ce0e0f75d63a5002afb9bdeb1ddaad857bed9
  Stored in directory: /root/.cache/pip/wheels/65/4f/35/5057db0249224e9ab55a513fa6b79451473ceb7713017823c3
Successfully built fasttext


# **RNN using fasttext embeddings**

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.vec.gz
!gunzip cc.ar.300.vec.gz



--2025-06-08 19:27:55--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.vec.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 18.173.166.31, 18.173.166.51, 18.173.166.74, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.173.166.31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1272365870 (1.2G) [binary/octet-stream]
Saving to: ‘cc.ar.300.vec.gz’

cc.ar.300.vec.gz    100%[===================>]   1.18G  49.8MB/s    in 22s     

2025-06-08 19:28:17 (55.9 MB/s) - ‘cc.ar.300.vec.gz’ saved [1272365870/1272365870]



In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [ ]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['preprocessed_question_body'])
sequences = tokenizer.texts_to_sequences(df['preprocessed_question_body'])

In [ ]:
max_seq_length = max(len(seq) for seq in sequences)
X = pad_sequences(sequences, maxlen=max_seq_length)

In [ ]:
embedding_index = {}
with open("cc.ar.300.vec", encoding='utf-8') as f:
    next(f)
    for line in f:
        values = line.rstrip().split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = vector

In [ ]:
embedding_dim = 300
vocab_size = len(tokenizer.word_index) + 1
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, idx in tokenizer.word_index.items():
    if idx < vocab_size:
        vector = embedding_index.get(word)
        if vector is not None:
            embedding_matrix[idx] = vector

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df['name_ar'])
y = to_categorical(y)
num_classes = y.shape[1]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim,
                    weights=[embedding_matrix], input_length=max_seq_length, trainable=False))
model.add(SimpleRNN(64))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(num_classes, activation='softmax'))


In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()
model.fit(X_train, y_train, epochs=10, batch_size=16)


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 71, 300)           10878600  
                                                                 
 simple_rnn (SimpleRNN)      (None, 64)                23360     
                                                                 
 dense (Dense)               (None, 32)                2080      
                                                                 
 dropout (Dropout)           (None, 32)                0         
                                                                 
 dense_1 (Dense)             (None, 5)                 165       
                                                                 
Total params: 10,904,205
Trainable params: 25,605
Non-trainable params: 10,878,600
_________________________________________________________________
Epoch 1/10
1500/1500 [=================

In [ ]:
from sklearn.metrics import classification_report, accuracy_score


loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")


y_pred_probs = model.predict(X_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_test, axis=1)


print("\nClassification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=le.classes_))


188/188 [==============================] - 4s 19ms/step - loss: 0.5807 - accuracy: 0.8140
Test Accuracy: 0.8140
188/188 [==============================] - 3s 17ms/step

Classification Report:
                       precision    recall  f1-score   support

          الطب النفسي       0.68      0.85      0.76      1212
                تغذية       0.86      0.74      0.79       882
جراحة العظام والمفاصل       0.90      0.81      0.85      1617
             طب اسنان       0.87      0.86      0.86      1243
              طب عيون       0.79      0.80      0.79      1046

             accuracy                           0.81      6000
            macro avg       0.82      0.81      0.81      6000
         weighted avg       0.82      0.81      0.82      6000



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Reshape, SimpleRNN, Dense, Dropout


In [ ]:
EXPECTED_DIM = 768


arabert_embeddings = pd.read_csv('arabert_no_stopwords_embeddings.csv', header=None)


valid_rows = []
for i, row in arabert_embeddings[0].items():
    vec = np.fromstring(row, sep=',')
    if vec.shape[0] == EXPECTED_DIM:
        valid_rows.append(vec)


X = np.array(valid_rows, dtype=np.float32)
print("Embedding shape:", X.shape)

<ipython-input-41-758ad1bc5d83>:9: DeprecationWarning: string or file could not be read to its end due to unmatched data; this will raise a ValueError in the future.
  vec = np.fromstring(row, sep=',')


Embedding shape: (30000, 768)


In [ ]:

labels = df['name_ar']
le = LabelEncoder()
y = le.fit_transform(labels)
y = to_categorical(y)
num_classes = y.shape[1]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:

model = Sequential()
model.add(Input(shape=(X.shape[1],)))
model.add(Reshape((X.shape[1], 1)))
model.add(SimpleRNN(64))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(num_classes, activation='softmax'))

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()
model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape_1 (Reshape)         (None, 768, 1)            0         
                                                                 
 simple_rnn_2 (SimpleRNN)    (None, 64)                4224      
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dropout_2 (Dropout)         (None, 32)                0         
                                                                 
 dense_5 (Dense)             (None, 5)                 165       
                                                                 
Total params: 6,469
Trainable params: 6,469
Non-trainable params: 0
_________________________________________________________________
Epoch 1/10
750/750 [==============================] 

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")



188/188 [==============================] - 15s 78ms/step
Accuracy:  0.7560
Precision: 0.7635
Recall:    0.7560
F1 Score:  0.7567


# **LSTM Fasttext**

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['preprocessed_question_body'].fillna(''))
sequences = tokenizer.texts_to_sequences(df['preprocessed_question_body'].fillna(''))
max_len = max(len(seq) for seq in sequences)
X_padded = pad_sequences(sequences, maxlen=max_len, padding='post', value=0)

vocab_size = len(tokenizer.word_index) + 1

le = LabelEncoder()
y = le.fit_transform(df['name_ar'])
y = to_categorical(y)
num_classes = y.shape[1]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_padded, y, test_size=0.2, random_state=42)

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim,
                    weights=[embedding_matrix], input_length=max_len, trainable=False)) # Use max_len here
model.add(LSTM(64))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(num_classes, activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
out_of_bounds_high = X_train[X_train >= vocab_size]
if out_of_bounds_high.size > 0:
    print(f"Error: Values >= vocab_size found in X_train! Max value: {np.max(out_of_bounds_high)}")
    high_indices = np.where(X_train >= vocab_size)
    print("Indices where out of bounds high values are found:", high_indices)
neg_indices = np.where(X_train < 0)
if neg_indices[0].size > 0:
    print(f"Error: Negative values found in X_train! Min value: {np.min(X_train)}")
    print("Indices where negative values are found:", neg_indices)


model.fit(X_train, y_train, epochs=10, batch_size=16, verbose=1)

Model: "sequential_9"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_7 (Embedding)     (None, 84, 300)           10878600  
                                                                 
 lstm_6 (LSTM)               (None, 64)                93440     
                                                                 
 dense_18 (Dense)            (None, 32)                2080      
                                                                 
 dropout_9 (Dropout)         (None, 32)                0         
                                                                 
 dense_19 (Dense)            (None, 5)                 165       
                                                                 
Total params: 10,974,285
Trainable params: 95,685
Non-trainable params: 10,878,600
_________________________________________________________________
Checking X_train values before traini

In [ ]:

from sklearn.metrics import precision_score, recall_score, f1_score


y_pred_probs_lstm_ft = model.predict(X_test)
y_pred_classes_lstm_ft = np.argmax(y_pred_probs_lstm_ft, axis=1)
y_true_classes_lstm_ft = np.argmax(y_test, axis=1)

acc_lstm_ft = accuracy_score(y_true_classes_lstm_ft, y_pred_classes_lstm_ft)
precision_lstm_ft = precision_score(y_true_classes_lstm_ft, y_pred_classes_lstm_ft, average='weighted')
recall_lstm_ft = recall_score(y_true_classes_lstm_ft, y_pred_classes_lstm_ft, average='weighted')
f1_lstm_ft = f1_score(y_true_classes_lstm_ft, y_pred_classes_lstm_ft, average='weighted')

print("\nEvaluation for LSTM FastText:")
print(f"Accuracy:  {acc_lstm_ft:.4f}")
print(f"Precision: {precision_lstm_ft:.4f}")
print(f"Recall:    {recall_lstm_ft:.4f}")
print(f"F1 Score:  {f1_lstm_ft:.4f}")

188/188 [==============================] - 12s 56ms/step

Evaluation for LSTM FastText:
Accuracy:  0.2697
Precision: 0.2798
Recall:    0.2697
F1 Score:  0.1148


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# LSTM **ArabBert**

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
EXPECTED_DIM = 768

arabert_embeddings_df = pd.read_csv('arabert_no_stopwords_embeddings.csv', header=None)

valid_vectors = []
for vec_str in arabert_embeddings_df[0]:
    if isinstance(vec_str, str) and vec_str.strip():
        try:
            vec = np.fromstring(vec_str, sep=',')
            if vec.shape[0] == EXPECTED_DIM:
                valid_vectors.append(vec)
            else:
                print(f"Warning: Skipping vector with unexpected shape {vec.shape}. Expected shape ({EXPECTED_DIM},). String: {vec_str[:50]}...") # print first 50 chars
        except Exception as e:
            print(f"Error parsing vector string: {vec_str[:50]}... Error: {e}")

if not valid_vectors:
    raise ValueError("No valid vectors found after parsing and filtering.")

X = np.stack(valid_vectors)

X = X.astype('float32')

<ipython-input-67-ea337c733b1c>:13: DeprecationWarning: string or file could not be read to its end due to unmatched data; this will raise a ValueError in the future.
  vec = np.fromstring(vec_str, sep=',')


In [ ]:
le = LabelEncoder()
y = le.fit_transform(df['name_ar'])
y = to_categorical(y)
num_classes = y.shape[1]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

In [ ]:
model = Sequential()
model.add(LSTM(64, input_shape=(1, X_train.shape[2])))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(num_classes, activation='softmax'))

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()
model.fit(X_train, y_train, epochs=10, batch_size=16, verbose=1)

Model: "sequential_10"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_7 (LSTM)               (None, 64)                213248    
                                                                 
 dense_20 (Dense)            (None, 32)                2080      
                                                                 
 dropout_10 (Dropout)        (None, 32)                0         
                                                                 
 dense_21 (Dense)            (None, 5)                 165       
                                                                 
Total params: 215,493
Trainable params: 215,493
Non-trainable params: 0
_________________________________________________________________
Epoch 1/10
1500/1500 [==============================] - 22s 12ms/step - loss: 0.5991 - accuracy: 0.7950
Epoch 2/10
1500/1500 [==============================] - 16s 10ms/step - los

In [ ]:

from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.naive_bayes import GaussianNB
from keras.utils import to_categorical
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout, Input, Reshape, LSTM
import gc

y_pred_probs_lstm_arabert = model.predict(X_test)
y_pred_classes_lstm_arabert = np.argmax(y_pred_probs_lstm_arabert, axis=1)
y_true_classes_lstm_arabert = np.argmax(y_test, axis=1)

acc_lstm_arabert = accuracy_score(y_true_classes_lstm_arabert, y_pred_classes_lstm_arabert)
precision_lstm_arabert = precision_score(y_true_classes_lstm_arabert, y_pred_classes_lstm_arabert, average='weighted')
recall_lstm_arabert = recall_score(y_true_classes_lstm_arabert, y_pred_classes_lstm_arabert, average='weighted')
f1_lstm_arabert = f1_score(y_true_classes_lstm_arabert, y_pred_classes_lstm_arabert, average='weighted')

print("\nEvaluation for LSTM AraBERT:")
print(f"Accuracy:  {acc_lstm_arabert:.4f}")
print(f"Precision: {precision_lstm_arabert:.4f}")
print(f"Recall:    {recall_lstm_arabert:.4f}")
print(f"F1 Score:  {f1_lstm_arabert:.4f}")


188/188 [==============================] - 2s 5ms/step

Evaluation for LSTM AraBERT:
Accuracy:  0.8623
Precision: 0.8685
Recall:    0.8623
F1 Score:  0.8636


# **Pre trained models (Fine tuning )**

# AraBERT

In [ ]:
df = pd.read_csv('sampled_30k.csv')

In [ ]:
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['name_ar'])
num_classes = len(label_encoder.classes_)

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['question_body'], df['label_encoded'], test_size=0.2, random_state=42
)

In [ ]:
model_name = "aubmindlab/bert-base-arabertv2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_classes)

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/720k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)

In [ ]:
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': list(train_labels)
})
test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': list(test_labels)
})

In [ ]:
training_args = TrainingArguments(
    output_dir="./arabert_output",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer
)

<ipython-input-18-1bb26a8c5282>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()


Step,Training Loss
50,1.641400
100,1.560200
150,1.363100
200,1.116400
250,0.936500
300,0.787000
350,0.745100
400,0.672900
450,0.632700
500,0.619300


TrainOutput(global_step=18000, training_loss=0.4032127438121372, metrics={'train_runtime': 2336.4721, 'train_samples_per_second': 30.816, 'train_steps_per_second': 7.704, 'total_flos': 4736126564352000.0, 'train_loss': 0.4032127438121372, 'epoch': 3.0})

In [18]:
y_pred_arabert_classification = trainer.predict(test_dataset).predictions.argmax(axis=1)

y_true_arabert_classification = test_labels


accuracy_arabert_classification = accuracy_score(y_true_arabert_classification, y_pred_arabert_classification)
precision_arabert_classification = precision_score(y_true_arabert_classification, y_pred_arabert_classification, average='weighted')
recall_arabert_classification = recall_score(y_true_arabert_classification, y_pred_arabert_classification, average='weighted')
f1_arabert_classification = f1_score(y_true_arabert_classification, y_pred_arabert_classification, average='weighted')


print("\nAraBERT Classification Metrics:")
print(f"Accuracy:  {accuracy_arabert_classification:.4f}")
print(f"Precision: {precision_arabert_classification:.4f}")
print(f"Recall:    {recall_arabert_classification:.4f}")
print(f"F1 Score:  {f1_arabert_classification:.4f}")


AraBERT Classification Metrics:
Accuracy:  0.8820
Precision: 0.8834
Recall:    0.8820
F1 Score:  0.8824


# **Pre-trained modesl (fine-tuning)**

# GPT2

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch


df = pd.read_csv('sampled_30k.csv')


label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['name_ar'])
num_classes = len(label_encoder.classes_)


train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['question_body'], df['label_encoded'], test_size=0.2, random_state=42
)


model_name = "akhooli/gpt2-small-arabic"
tokenizer = AutoTokenizer.from_pretrained(model_name)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_classes)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at akhooli/gpt2-small-arabic and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)

train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': list(train_labels)
})
test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': list(test_labels)
})


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1).numpy()
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:



training_args = TrainingArguments(
    output_dir="./gpt2_classification_output",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics  # ✅ Add metrics
)


trainer.train()



<ipython-input-11-36bae75a41ca>:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,2.188100
20,1.562900
30,1.730600
40,1.749800
50,1.575500
60,1.819700
70,1.955400
80,1.779400
90,1.917600
100,1.939500


TrainOutput(global_step=36000, training_loss=0.49356040421474606, metrics={'train_runtime': 3433.6617, 'train_samples_per_second': 20.969, 'train_steps_per_second': 10.484, 'total_flos': 4703468912640000.0, 'train_loss': 0.49356040421474606, 'epoch': 3.0})

In [17]:
eval_results = trainer.evaluate()


print("GPT-2 Evaluation Metrics:")
print(f"Accuracy:  {eval_results['eval_accuracy']:.4f}")
print(f"Precision: {eval_results['eval_precision']:.4f}")
print(f"Recall:    {eval_results['eval_recall']:.4f}")
print(f"F1 Score:  {eval_results['eval_f1']:.4f}")


GPT-2 Evaluation Metrics:
Accuracy:  0.8820
Precision: 0.8832
Recall:    0.8793
F1 Score:  0.8810
